# Model Training

In this notebook, we will ask you a series of questions regarding model selection. Based on your responses, we will ask you to create the ML models that you've chosen. 

The bonus step is completely optional, but if you provide a sufficient third machine learning model in this project, we will add `1000` points to your Kahoot leaderboard score.

**Note**: Use the dataset that you've created in your previous data transformation step (not the original model).

## Questions
Is this a classification or regression task?  

Answer here: 
This is a classification task.

Are you predicting for multiple classes or binary classes?  

Answer here:    
It’s a binary classification (fraudulent or not fraudulent, where fraud = 1, not fraud = 0).

Given these observations, which 2 (or possibly 3) machine learning models will you choose?  

List your models here:  
1. Logistic Regression
2. Random Forest Classifier
3. XGBoost Classifier (Bonus)

## First Model

Using the first model that you've chosen, implement the following steps.

### 1) Create a train-test split

Use your cleaned and transformed dataset to divide your features and labels into training and testing sets. Make sure you’re only using numeric or properly encoded features.  

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the cleaned dataset (replace with the correct local path)
df = pd.read_csv('../data/engineered_transactions.csv')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Check shape and class distribution
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_test distribution:\n", y_test.value_counts(normalize=True))


X_train shape: (800000, 14)
X_test shape: (200000, 14)
y_train distribution:
 isFraud
0    0.998703
1    0.001298
Name: proportion, dtype: float64
y_test distribution:
 isFraud
0    0.998705
1    0.001295
Name: proportion, dtype: float64


### 2) Search for best hyperparameters
Use tools like GridSearchCV, RandomizedSearchCV, or model-specific tuning functions to find the best hyperparameters for your first model.

In [3]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

# Step 1: Sample a smaller training set to speed things up
X_sample = X_train.sample(n=50000, random_state=42)
y_sample = y_train.loc[X_sample.index]

# Step 2: Define the pipeline with SMOTE and Logistic Regression
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear'))
])

# Step 3: Define hyperparameter space
param_dist = {
    'logreg__C': [0.01, 0.1, 1, 10],
    'logreg__penalty': ['l1', 'l2']
}

# Step 4: Set up RandomizedSearchCV
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=5,  # Only try 5 random combinations
    scoring='f1',
    cv=3,  # 3-fold CV
    n_jobs=-1,  # Use all cores
    random_state=42,
    verbose=1
)

# Step 5: Fit on the sampled training set
random_search.fit(X_sample, y_sample)

# Step 6: Output best model
best_model = random_search.best_estimator_
print("Best hyperparameters:", random_search.best_params_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best hyperparameters: {'logreg__penalty': 'l1', 'logreg__C': 0.1}


### 3) Train your model
Select the model with best hyperparameters and generate predictions on your test set. Evaluate your models accuracy, precision, recall, and sensitivity.  

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set
y_pred = best_model.predict(X_test)

# Evaluation metrics
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99    199741
           1       0.05      0.97      0.09       259

    accuracy                           0.98    200000
   macro avg       0.52      0.97      0.54    200000
weighted avg       1.00      0.98      0.99    200000

Confusion Matrix:
 [[194903   4838]
 [     7    252]]


Model 1: Logistic Regression with SMOTE (on 50,000 sample)  
To address class imbalance, I applied SMOTE to a 50,000-sample subset of the training data and tuned a Logistic Regression model using RandomizedSearchCV.

Best Hyperparameters:
penalty='l1', C=0.1

Model Evaluation on Test Set:  
Accuracy: 98%  
Fraud Recall (class 1): 97%  
Fraud Precision (class 1): 5% 

Interpretation:
The model is highly effective at identifying most fraudulent transactions (high recall), but also flags many non-fraud cases as fraud (low precision), which is a common trade-off when using oversampling like SMOTE.

## Second Model

Create a second machine learning object and rerun steps (2) & (3) on this model. Compare accuracy metrics between these two models. Which handles the class imbalance more effectively?

Create as many code-blocks as needed.

### (Bonus/Optional) Third Model

Create a third machine learning model and rerun steps (2) & (3) on this model. Which model has the best predictive capabilities? 

Create as many code-blocks as needed.